## 权重衰减（Weight Decay）套用模板

---

### 一、核心概念一句话

> **权重衰减 = 在优化器中设置 `weight_decay`，自动把权重拉向 0，防止过拟合。**

---

### 二、标准模板（直接套用）

```python
import torch
import torch.nn as nn

# ============ 1. 定义模型 ============
net = nn.Sequential(
    nn.Linear(num_inputs, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

# ============ 2. 初始化参数 ============
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.constant_(m.bias, 0)
net.apply(init_weights)

# ============ 3. 损失函数 ============
loss = nn.MSELoss()                    # 回归用 MSE
# loss = nn.CrossEntropyLoss()         # 分类用交叉熵

# ============ 4. 优化器（🔑 核心：weight_decay） ============
trainer = torch.optim.SGD([
    {"params": net[0].weight, "weight_decay": 3},   # ✅ 权重衰减
    {"params": net[0].bias}                           # ❌ 偏置不衰减
    # ... 其他层同理
], lr=0.003)

# 更简单的写法（所有可学习参数统一衰减）：
trainer = torch.optim.SGD(net.parameters(), lr=0.003, weight_decay=3)

# AdamW 写法（更推荐）：
trainer = torch.optim.AdamW(net.parameters(), lr=0.001, weight_decay=0.01)

# ============ 5. 训练循环 ============
for epoch in range(num_epochs):
    for X, y in data_iter:
        trainer.zero_grad()
        l = loss(net(X), y)
        l.backward()
        trainer.step()
```

---

### 三、两种分组方式对比

```python
# 方式 A：精细分组（偏置不衰减）
trainer = torch.optim.SGD([
    {"params": net[0].weight, "weight_decay": wd},
    {"params": net[0].bias}                            # 默认 weight_decay=0
], lr=0.003)

# 方式 B：统一衰减（所有参数都衰减，简单粗暴）
trainer = torch.optim.SGD(net.parameters(), lr=0.003, weight_decay=wd)
```

| | 方式 A（分组） | 方式 B（统一） |
|------|---------------|----------------|
| `weight` | ✅ 衰减 | ✅ 衰减 |
| `bias` | ❌ 不衰减 | ✅ 衰减 |
| 适用 | 精细控制 | 快速实验 |

---

### 四、不同优化器的 weight_decay 写法

```python
# SGD
optim.SGD(params, lr=0.01, weight_decay=1e-4)

# SGD + 动量
optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=1e-4)

# Adam（⚠️ 用 AdamW 更正确）
optim.Adam(params, lr=0.001, weight_decay=1e-4)

# AdamW（✅ 推荐：解耦了衰减和自适应学习率）
optim.AdamW(params, lr=0.001, weight_decay=1e-2)

# RMSprop
optim.RMSprop(params, lr=0.01, weight_decay=1e-4)
```

---

### 五、weight_decay 取值指南

```python
# 经验范围
weight_decay = 1e-4   # 很弱的正则化，大模型常用
weight_decay = 1e-3   # 中等正则化
weight_decay = 1e-2   # 较强正则化（你例子中的 3 是极端情况）
weight_decay = 0      # 不用正则化

# 选择原则
# 过拟合严重 → 调大（如 1e-2）
# 欠拟合     → 调小（如 1e-5）或设为 0
# 正常范围   → 1e-4 ~ 1e-3
```

---

### 六、完整训练函数模板

```python
def train_model(wd):
    net = nn.Sequential(nn.Linear(num_inputs, 1))
    nn.init.normal_(net[0].weight, 0, 0.01)
    nn.init.constant_(net[0].bias, 0)

    loss = nn.MSELoss()
    trainer = torch.optim.SGD([
        {"params": net[0].weight, "weight_decay": wd},
        {"params": net[0].bias}
    ], lr=0.003)

    for epoch in range(100):
        for X, y in data_iter:
            trainer.zero_grad()
            l = loss(net(X), y)
            l.backward()
            trainer.step()

    # 返回测试损失
    with torch.no_grad():
        return loss(net(test_X), test_y).item()

# 对比有无权重衰减
loss_no_wd = train_model(0)       # 过拟合
loss_with_wd = train_model(3)     # 泛化好
```

---

### 七、手写版 ↔ 简洁版对应

```
手写版：                                 简洁版：
────────────────────────────────────────────────────
l = MSE + λ/2 * Σw²                     weight_decay = λ   ← 优化器里设
w.grad = ∇MSE + λ*w                     trainer.step() 自动完成
w -= lr * (∇MSE + λ*w)                  无需手动加惩罚项
```

---

### 八、速查清单

| 步骤 | 关键代码 |
|------|----------|
| 模型 | `nn.Sequential(nn.Linear(d_in, d_out))` |
| 损失 | `nn.MSELoss()` 或 `nn.CrossEntropyLoss()` |
| 优化器 | `optim.SGD(params, lr, weight_decay=wd)` |
| 训练 | `zero_grad() → loss → backward() → step()` |
| 偏置 | 通常**不**加 weight_decay |
| 调参 | `1e-4`（弱）→ `1e-3`（中）→ `1e-2`（强） |